In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

import subprocess
import re

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

### coarse peaks

In [ ]:
# auto_peaks

### justin chromhmm

In [ ]:
# final_peaks

### chromHMM

In [ ]:
indir = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v6_Data'
name = 'Ernst_stateannotate_YS3v6.FullArchR_NamedCluster_DiffPeaks.table.txt.gz'
peaks = pd.read_csv(f'{indir}/{name}', sep='\t').sort_values(by=['seqnames', 'start'])
peaks = peaks.dropna()
peaks.shape

In [ ]:
auto = [f'chr{x}' for x in range(1,23)]
mask = [x in auto for x in peaks['seqnames']]
auto_peaks = peaks[mask]
auto_peaks['chr_num'] = [int(x.split('chr')[1]) for x in auto_peaks['seqnames']]
auto_peaks.sort_values(by=['chr_num', 'start'], inplace=True)
auto_peaks.index = [f'{x}_{y}' for x,y in zip(auto_peaks['seqnames'], auto_peaks['start'])]
auto_peaks.shape

In [ ]:
justin = pd.read_csv(f'{projdir}/csv/clusters/justin_v7.csv', sep=',', index_col=0)
rename_dict = dict(zip(justin['Old_Specific'], justin['New_Specific']))

In [ ]:
def categorize_chromhmm_state(state):
    if re.match(r'^(TSS|PromF|BivProm)', state):
        return 'Promoter'
    elif re.match(r'^(EnhA|EnhWk|TxEnh)', state):
        return 'Enhancer'
    elif re.match(r'^(Tx|TxEx)', state):
        return 'Transcribed'
    elif re.match(r'^(Acet|DNase)', state):
        return 'ActiveRegulatory'
    elif re.match(r'^(ReprPC)', state):
        return 'Repressed'
    elif re.match(r'^(HET|znf)', state):
        return 'Heterochromatin'
    elif re.match(r'^(Quies)', state):
        return 'Quiescent'
    elif re.match(r'^(GapArtf)', state):
        return 'Artifact'
    else:
        return 'Other'

In [ ]:
auto_peaks['cluster_v7'] = [rename_dict.get(x) for x in auto_peaks['Sample']]
auto_peaks['coarse_v7'] = [x.split('_')[0] for x in auto_peaks['cluster_v7']]
auto_peaks['broad_class'] = [categorize_chromhmm_state(x) for x in auto_peaks['UniversalChromHMMState']]

In [ ]:
auto_peaks.head()

In [ ]:
outdir = f'{projdir}/atac/peaks/chromhmm'
auto_peaks.to_csv(f'{outdir}/justin.chromhmm.header.bed.gz', sep='\t', header=True, index=None)

In [ ]:
out = f'{projdir}/atac/peaks/chromhmm/justin.chromhmm.bed'
auto_peaks.to_csv(out, sep='\t', header=None, index=None)
subprocess.run(["bgzip", '-f', out], check=True)
subprocess.run(["tabix", "-p", "bed", f"{out}.gz"], check=True)
auto_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
outdir = f'{projdir}/atac/peaks/chromhmm/coarse'

tmp_clusts = auto_peaks['coarse_v7'].unique()
for s in tmp_clusts:
    tmp_peaks = auto_peaks[auto_peaks['coarse_v7'] == s]
    out = f'{outdir}/{s}.bed'
    tmp_peaks.to_csv(out, sep='\t', header=None, index=None)
    subprocess.run(["bgzip", '-f', out], check=True)
    subprocess.run(["tabix", "-p", "bed", f"{out}.gz"], check=True)
    tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
outdir = f'{projdir}/atac/peaks/chromhmm/cluster'

tmp_clusts = auto_peaks['cluster_v7'].unique()
for s in tmp_clusts:
    tmp_peaks = auto_peaks[auto_peaks['cluster_v7'] == s]
    out = f'{outdir}/{s}.bed'
    tmp_peaks.to_csv(out, sep='\t', header=None, index=None)
    subprocess.run(["bgzip", '-f', out], check=True)
    subprocess.run(["tabix", "-p", "bed", f"{out}.gz"], check=True)
    tmp_peaks.to_csv(out, sep='\t', header=None, index=None)

### deduplicate (do I need this?)

In [ ]:
final_peaks = auto_peaks[~auto_peaks.index.duplicated()] 
final_peaks.shape

In [ ]:
final_peaks.head()

In [ ]:
out = f'{outdir}/justin.chromhmm.dedup.bed'
final_peaks.to_csv(out, sep='\t', header=None, index=None)
subprocess.run(["bgzip", '-f', out], check=True)
subprocess.run(["tabix", "-p", "bed", f"{out}.gz"], check=True)
final_peaks.to_csv(out, sep='\t', header=None, index=None)

In [ ]:
!date